# Retrieval

Mirrors the structure of the LangChain "Chat with Your Data" retrieval lecture (`04_retrieval.ipynb`), applied to our real Bahraini legal corpus (v2 vector store) instead of lecture sample data. Everything before this notebook (scraping, cleaning, chunking, embedding) is unchanged — this picks up from the vector store we already built.

In [ ]:
!pip install -q langchain-huggingface langchain-chroma langchain-core langchain-text-splitters langchain-community langchain-classic langchain-openai sentence-transformers transformers chromadb


In [ ]:
import os
from google.colab import userdata

os.environ["OPENROUTER_API_KEY"] = userdata.get("OPENROUTER_API_KEY")


## Vectorstore retrieval

In [ ]:
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from google.colab import drive
import torch
drive.mount("/content/drive")

persist_directory = "/content/drive/MyDrive/law_chatbot_chroma_v2"
device = "cuda" if torch.cuda.is_available() else "cpu"
embedding = HuggingFaceEmbeddings(model_name="BAAI/bge-m3", model_kwargs={"device": device})
vectordb = Chroma(persist_directory=persist_directory, embedding_function=embedding)
print(vectordb._collection.count())


### Similarity Search

Before searching the full ~25,000-passage corpus, let's see the basic mechanic on a tiny, real 3-document index — one labor-law article, one lease-law article, and one custody-law article, pulled straight from our own corpus. Same teaching moment as the lecture's toy mushroom example, but real Bahraini statute text instead of sample data.

In [ ]:
toy_texts = [
    "المادة (105) يجوز للعامل انهاء عقد العمل دون اخطار في اي من الحالتين التاليتين: 1) اعتداء صاحب العمل او من ينوب عنه علي العامل، اثناء العمل او بسببه، بقول او فعل معاقب عليه قانونا. 2) ارتكاب صاحب العمل او من يمثله امرا مخلا بالاداب نحو العامل او احد افراد اسرته. ويعتبر انهاء العقد في هاتين الحالتين بمثابة فصل تعسفي من جانب صاحب العمل.",
    "المادة (4) ا) يجب تحديد مدة الايجار، فاذا عقد الايجار دون اتفاق علي مدة، او عقد لمدة غير محددة، او تعذر اثبات مدته المدعاة، اعتبر العقد منعقدا للمدة المحددة لدفع الاجرة. ب) يجب تحديد مقدار الاجرة في العقد، فاذا لم يتفق الطرفان علي مقدارها او كيفية تقديرها او تعذر اثبات مقدارها، وجب اعتبار اجرة المثل وقت ابرام العقد، ويراعي في تقديرها حالة العين ومساحتها والغرض المعدة له والاجرة السائدة في منطقتها.",
    "المادة (125) 1) وفقا للفقه السني اذا بلغ الذكر خمس عشرة سنة، او بلغت الانثي سبع عشرة سنة ولم تتزوج ولم يدخل بها الزوج، فلكل منهما الخيار في الانضمام الي من يشاء من ابويه او ممن له الحق في حضانته.",
]
toy_metadatas = [
    {"source": "lloc", "doc_id": "K3612", "article_no": "105", "topic": "labor"},
    {"source": "lloc", "doc_id": "K2714", "article_no": "4", "topic": "lease"},
    {"source": "lloc", "doc_id": "K1917", "article_no": "125", "topic": "custody"},
]
smalldb = Chroma.from_texts(toy_texts, embedding=embedding, metadatas=toy_metadatas)

toy_question = "هل يجوز فصل الموظف بدون سابق انذار؟"
smalldb.similarity_search(toy_question, k=2)


In [ ]:
smalldb.max_marginal_relevance_search(toy_question, k=2, fetch_k=3)

### Addressing Diversity: Maximum marginal relevance (MMR)

In [ ]:
question = "ما هي شروط فسخ عقد الايجار؟"
docs_ss = vectordb.similarity_search(question, k=3)
docs_mmr = vectordb.max_marginal_relevance_search(question, k=3)

print("--- plain similarity search ---")
for d in docs_ss:
    print(d.page_content[:100], "\n")

print("--- MMR ---")
for d in docs_mmr:
    print(d.page_content[:100], "\n")

### Addressing Specificity: working with metadata

In [ ]:
docs = vectordb.similarity_search(
    question,
    k=3,
    filter={"source": "lloc"},  # legislation only
)
for d in docs:
    print(d.metadata)

### Addressing Specificity: working with metadata using self-query retriever

In [ ]:
!pip install -q -U langchain-community langchain-classic


In [ ]:
from langchain_openai import ChatOpenAI
from langchain_classic.retrievers.self_query.base import SelfQueryRetriever
from langchain_classic.retrievers.self_query.chroma import ChromaTranslator
from langchain_classic.chains.query_constructor.base import AttributeInfo

metadata_field_info = [
    AttributeInfo(
        name="source",
        description="مصدر هذا المقطع، احد ثلاثة:\nlloc: التشريعات والقوانين\nsjc: احكام محكمة التمييز\nccb: احكام المحكمة الدستورية",
        type="string",
    ),
    AttributeInfo(
        name="doc_id",
        description="رمز القانون او رقم القضية والطعن الذي يعود اليه هذا المقطع",
        type="string",
    ),
]

llm = ChatOpenAI(
    model="nvidia/nemotron-3-ultra-550b-a55b:free",
    temperature=0,
    api_key=os.environ["OPENROUTER_API_KEY"],
    base_url="https://openrouter.ai/api/v1",
)

document_content_description = "نصوص قانونية بحرينية، مواد تشريعية واحكام قضائية"
retriever = SelfQueryRetriever.from_llm(
    llm,
    vectordb,
    document_content_description,
    metadata_field_info,
    verbose=True,
    structured_query_translator=ChromaTranslator(),
)

In [ ]:
docs = retriever.invoke(question)
for d in docs:
    print(d.metadata)

### Additional tricks: compression

Information relevant to a question can be buried inside a much larger retrieved passage. **This matters more for us than it did in the lecture**: our v2 chunks can run to 180,000+ characters for consolidated multi-case judgments, so contextual compression isn't just a lecture exercise here — it genuinely trims a giant retrieved chunk down to only the sentences relevant to the question before it reaches the LLM.

In [ ]:
from langchain_classic.retrievers import ContextualCompressionRetriever
from langchain_classic.retrievers.document_compressors import LLMChainExtractor

def pretty_print_docs(docs):
    print(f"\n{'-' * 100}\n".join([f"Document {i+1}:\n\n" + d.page_content for i, d in enumerate(docs)]))

compressor = LLMChainExtractor.from_llm(llm)
compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor,
    base_retriever=vectordb.as_retriever(search_kwargs={"k": 3}),
)

In [ ]:
compression_question = "ما هي مدة الايجار اذا لم يحدد الطرفان مدة العقد؟"
compressed_docs = compression_retriever.invoke(compression_question)
pretty_print_docs(compressed_docs)

### Combining various techniques

In [ ]:
compression_retriever_mmr = ContextualCompressionRetriever(
    base_compressor=compressor,
    base_retriever=vectordb.as_retriever(search_type="mmr", search_kwargs={"k": 3}),
)
compressed_docs_mmr = compression_retriever_mmr.invoke(compression_question)
pretty_print_docs(compressed_docs_mmr)

### Other types of retrieval

It's worth noting that the vector store isn't the only way to retrieve documents. LangChain's retriever abstraction includes lexical/statistical methods too, such as TF-IDF or SVM.

In [ ]:
from langchain_community.retrievers import TFIDFRetriever

all_texts = vectordb.get()["documents"]
tfidf_retriever = TFIDFRetriever.from_texts(all_texts)

docs_tfidf = tfidf_retriever.invoke(question)
docs_tfidf[0].page_content[:200] if docs_tfidf else "no results"

**Note:** `SVMRetriever.from_texts` embeds every document it's given from scratch on the CPU, which we measured at well over 10 minutes for even a few dozen of our passages on this machine — impractically slow, and pointless besides, since Chroma already computed and stored an embedding for every passage in the corpus. So instead of re-embedding, we pull a sample of precomputed vectors straight out of the vector store and build the SVM index from those directly. The only embedding call left at query time is the single query itself.

In [ ]:
import numpy as np
from langchain_community.retrievers import SVMRetriever

sample = vectordb.get(include=["embeddings", "documents"], limit=300)
svm_texts = sample["documents"]
svm_index = np.array(sample["embeddings"])

svm_retriever = SVMRetriever(embeddings=embedding, index=svm_index, texts=svm_texts)

docs_svm = svm_retriever.invoke(question)
docs_svm[0].page_content[:300] if docs_svm else "no results"